<a href="https://colab.research.google.com/github/safwan-bhuiyan2/Dashboards/blob/main/Walmart-Sales-Data-Dashboard/Walmart_Sales_Data_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Walmart_customer_purchases.csv to Walmart_customer_purchases.csv


In [4]:
!pip install dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 17.4 MB/s eta 0:00:00
  Attempting uninstall: Werkzeug
    Found existing installation: Werkzeug 3.1.3
    Uninstalling Werkzeug-3.1.3:
      Successfully uninstalled Werkzeug-3.1.3
  Attempting uninstall: Flask
    Found existing installation: Flask 3.1.0
    Uninstalling Flask-3.1.0:
      Successfully uninstalled Flask-3.1.0


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.read_csv('Walmart_customer_purchases.csv')

In [ ]:
df.head()

,Customer_ID,Age,Gender,City,Category,Product_Name,Purchase_Date,Purchase_Amount,Payment_Method,Discount_Applied,Rating,Repeat_Customer
0,84607c1f-910c-44d5-b89f-e1ee06dd34c0,49,Female,New Cynthia,Electronics,Smartphone,2024-08-30,253.26,Cash on Delivery,No,1,Yes
1,f2a81712-a73e-4424-8b39-4c615a0bd4ea,36,Other,Cruzport,Clothing,T-Shirt,2024-12-21,73.19,Debit Card,Yes,1,No
2,da9be287-8b0e-4688-bccd-1a2cdd7567c6,52,Male,Jeffreytown,Beauty,Perfume,2024-12-26,125.62,Credit Card,Yes,1,No
3,50ec6932-3ac7-492f-9e55-4b148212f302,47,Female,Jenniferburgh,Electronics,Smartwatch,2024-11-04,450.32,Credit Card,No,2,Yes
4,8fdc3098-fc75-4b0f-983c-d8d8168c6362,43,Other,Kingshire,Electronics,Smartphone,2024-10-07,369.28,Credit Card,Yes,2,Yes


In [ ]:
df.columns

Index(['Customer_ID', 'Age', 'Gender', 'City', 'Category', 'Product_Name',
       'Purchase_Date', 'Purchase_Amount', 'Payment_Method',
       'Discount_Applied', 'Rating', 'Repeat_Customer'],
      dtype='object')

In [ ]:
# check for empty columns
df.isnull().sum()

,0
Customer_ID,0
Age,0
Gender,0
City,0
Category,0
Product_Name,0
Purchase_Date,0
Purchase_Amount,0
Payment_Method,0
Discount_Applied,0


In [ ]:
category_counts = df['Category'].value_counts()
print(category_counts)

Category
Electronics    12642
Home           12492
Beauty         12447
Clothing       12419
Name: count, dtype: int64


In [5]:
import pandas as pd
import numpy as np
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend for server environments
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import base64
from io import BytesIO

# Load the data directly from your CSV file
def load_data():
    df = pd.read_csv("Walmart_customer_purchases.csv")

    # Ensure Purchase_Date is datetime
    df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])

    # Convert Discount_Applied to boolean if it's not already
    if df['Discount_Applied'].dtype == 'object':
        df['Discount_Applied'] = df['Discount_Applied'].map({'True': True, 'False': False})

    # Convert Repeat_Customer to boolean if it's not already
    if df['Repeat_Customer'].dtype == 'object':
        df['Repeat_Customer'] = df['Repeat_Customer'].map({'True': True, 'False': False})

    return df

# Load the data
df = load_data()

# Initialize the Dash app
app = dash.Dash(__name__, suppress_callback_exceptions=True)
server = app.server

# Define colors
colors = {
    'background': '#F9F9F9',
    'text': '#333333',
    'walmart_blue': '#0071CE',
    'walmart_yellow': '#FFC220',
    'light_gray': '#EEEEEE'
}

# Create app layout
app.layout = html.Div(style={'backgroundColor': colors['background'], 'fontFamily': 'Arial, sans-serif'}, children=[
    html.Div(style={'backgroundColor': colors['walmart_blue'], 'padding': '20px', 'color': 'white'}, children=[
        html.H1('Walmart Purchase Analytics Dashboard', style={'textAlign': 'center'}),
    ]),

    html.Div(style={'padding': '20px'}, children=[
        html.Div(style={'display': 'flex', 'flexWrap': 'wrap', 'justifyContent': 'space-between'}, children=[
            # Date range filter
            html.Div(style={'width': '100%', 'marginBottom': '20px'}, children=[
                html.H4('Select Date Range:', style={'color': colors['text']}),
                dcc.DatePickerRange(
                    id='date-range',
                    min_date_allowed=df['Purchase_Date'].min().date(),
                    max_date_allowed=df['Purchase_Date'].max().date(),
                    start_date=df['Purchase_Date'].min().date(),
                    end_date=df['Purchase_Date'].max().date(),
                    style={'width': '100%'}
                ),
            ]),

            # Category dropdown filter
            html.Div(style={'width': '30%', 'marginBottom': '20px'}, children=[
                html.H4('Select Category:', style={'color': colors['text']}),
                dcc.Dropdown(
                    id='category-dropdown',
                    options=[{'label': 'All Categories', 'value': 'All'}] +
                            [{'label': cat, 'value': cat} for cat in sorted(df['Category'].unique())],
                    value='All',
                    clearable=False
                ),
            ]),

            # City dropdown filter
            html.Div(style={'width': '30%', 'marginBottom': '20px'}, children=[
                html.H4('Select City:', style={'color': colors['text']}),
                dcc.Dropdown(
                    id='city-dropdown',
                    options=[{'label': 'All Cities', 'value': 'All'}] +
                            [{'label': city, 'value': city} for city in sorted(df['City'].unique())],
                    value='All',
                    clearable=False
                ),
            ]),

            # Payment method dropdown filter
            html.Div(style={'width': '30%', 'marginBottom': '20px'}, children=[
                html.H4('Select Payment Method:', style={'color': colors['text']}),
                dcc.Dropdown(
                    id='payment-dropdown',
                    options=[{'label': 'All Payment Methods', 'value': 'All'}] +
                            [{'label': pm, 'value': pm} for pm in sorted(df['Payment_Method'].unique())],
                    value='All',
                    clearable=False
                ),
            ]),
        ]),

        # KPI Cards
        html.Div(style={'display': 'flex', 'justifyContent': 'space-between', 'marginBottom': '20px'}, children=[
            # Total Sales KPI
            html.Div(id='total-sales-card', style={
                'width': '23%',
                'padding': '15px',
                'backgroundColor': 'white',
                'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                'borderRadius': '5px',
                'textAlign': 'center'
            }),

            # Average Purchase KPI
            html.Div(id='avg-purchase-card', style={
                'width': '23%',
                'padding': '15px',
                'backgroundColor': 'white',
                'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                'borderRadius': '5px',
                'textAlign': 'center'
            }),

            # Unique Customers KPI
            html.Div(id='unique-customers-card', style={
                'width': '23%',
                'padding': '15px',
                'backgroundColor': 'white',
                'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                'borderRadius': '5px',
                'textAlign': 'center'
            }),

            # Total Orders KPI
            html.Div(id='total-orders-card', style={
                'width': '23%',
                'padding': '15px',
                'backgroundColor': 'white',
                'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                'borderRadius': '5px',
                'textAlign': 'center'
            }),
        ]),

        # Graphs section
        html.Div(style={'display': 'flex', 'flexWrap': 'wrap', 'justifyContent': 'space-between'}, children=[
            # Sales by Category Bar Chart
            html.Div(style={'width': '48%', 'marginBottom': '20px'}, children=[
                html.Div(style={
                    'backgroundColor': 'white',
                    'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                    'borderRadius': '5px',
                    'padding': '15px'
                }, children=[
                    html.H3('Sales by Category', style={'textAlign': 'center', 'color': colors['text']}),
                    dcc.Graph(id='category-sales-graph')
                ])
            ]),

            # Sales by Payment Method Pie Chart
            html.Div(style={'width': '48%', 'marginBottom': '20px'}, children=[
                html.Div(style={
                    'backgroundColor': 'white',
                    'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                    'borderRadius': '5px',
                    'padding': '15px'
                }, children=[
                    html.H3('Sales by Payment Method', style={'textAlign': 'center', 'color': colors['text']}),
                    dcc.Graph(id='payment-sales-pie')
                ])
            ]),

            # Monthly Sales Trend Line Chart
            html.Div(style={'width': '48%', 'marginBottom': '20px'}, children=[
                html.Div(style={
                    'backgroundColor': 'white',
                    'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                    'borderRadius': '5px',
                    'padding': '15px'
                }, children=[
                    html.H3('Monthly Sales Trend', style={'textAlign': 'center', 'color': colors['text']}),
                    dcc.Graph(id='monthly-sales-trend')
                ])
            ]),

            # Customer Gender Distribution Pie Chart
            html.Div(style={'width': '48%', 'marginBottom': '20px'}, children=[
                html.Div(style={
                    'backgroundColor': 'white',
                    'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                    'borderRadius': '5px',
                    'padding': '15px'
                }, children=[
                    html.H3('Customer Gender Distribution', style={'textAlign': 'center', 'color': colors['text']}),
                    dcc.Graph(id='gender-distribution-pie')
                ])
            ]),

            # Customer Age Distribution
            html.Div(style={'width': '48%', 'marginBottom': '20px'}, children=[
                html.Div(style={
                    'backgroundColor': 'white',
                    'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                    'borderRadius': '5px',
                    'padding': '15px'
                }, children=[
                    html.H3('Customer Age Distribution', style={'textAlign': 'center', 'color': colors['text']}),
                    dcc.Graph(id='age-distribution-hist')
                ])
            ]),

            # Word Cloud for Products
            html.Div(style={'width': '48%', 'marginBottom': '20px'}, children=[
                html.Div(style={
                    'backgroundColor': 'white',
                    'boxShadow': '0px 0px 10px rgba(0, 0, 0, 0.1)',
                    'borderRadius': '5px',
                    'padding': '15px'
                }, children=[
                    html.H3('Product Word Cloud', style={'textAlign': 'center', 'color': colors['text']}),
                    html.Img(id='wordcloud-image', style={'width': '100%'})
                ])
            ]),
        ]),
    ]),
])

# Define callback to update all components based on filters
@app.callback(
    [
        Output('total-sales-card', 'children'),
        Output('avg-purchase-card', 'children'),
        Output('unique-customers-card', 'children'),
        Output('total-orders-card', 'children'),
        Output('category-sales-graph', 'figure'),
        Output('payment-sales-pie', 'figure'),
        Output('monthly-sales-trend', 'figure'),
        Output('gender-distribution-pie', 'figure'),
        Output('age-distribution-hist', 'figure'),
        Output('wordcloud-image', 'src')
    ],
    [
        Input('date-range', 'start_date'),
        Input('date-range', 'end_date'),
        Input('category-dropdown', 'value'),
        Input('city-dropdown', 'value'),
        Input('payment-dropdown', 'value')
    ]
)
def update_dashboard(start_date, end_date, category, city, payment_method):
    # Convert string dates to datetime
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # Apply date range filter
    filtered_df = df[(df['Purchase_Date'] >= start_date) & (df['Purchase_Date'] <= end_date)]

    # Apply category filter if not 'All'
    if category != 'All':
        filtered_df = filtered_df[filtered_df['Category'] == category]

    # Apply city filter if not 'All'
    if city != 'All':
        filtered_df = filtered_df[filtered_df['City'] == city]

    # Apply payment method filter if not 'All'
    if payment_method != 'All':
        filtered_df = filtered_df[filtered_df['Payment_Method'] == payment_method]

    # Calculate KPIs
    total_sales = filtered_df['Purchase_Amount'].sum()
    avg_purchase = filtered_df['Purchase_Amount'].mean()
    unique_customers = filtered_df['Customer_ID'].nunique()
    total_orders = len(filtered_df)

    # Create KPI card contents
    total_sales_card = [
        html.H4('Total Sales', style={'margin': '0', 'color': colors['walmart_blue']}),
        html.H2(f'${total_sales:,.2f}', style={'margin': '10px 0', 'color': colors['text']}),
        html.P('Total revenue from all purchases', style={'margin': '0', 'color': '#888'})
    ]

    avg_purchase_card = [
        html.H4('Average Purchase', style={'margin': '0', 'color': colors['walmart_blue']}),
        html.H2(f'${avg_purchase:,.2f}', style={'margin': '10px 0', 'color': colors['text']}),
        html.P('Average amount per transaction', style={'margin': '0', 'color': '#888'})
    ]

    unique_customers_card = [
        html.H4('Unique Customers', style={'margin': '0', 'color': colors['walmart_blue']}),
        html.H2(f'{unique_customers:,}', style={'margin': '10px 0', 'color': colors['text']}),
        html.P('Total number of distinct customers', style={'margin': '0', 'color': '#888'})
    ]

    total_orders_card = [
        html.H4('Total Orders', style={'margin': '0', 'color': colors['walmart_blue']}),
        html.H2(f'{total_orders:,}', style={'margin': '10px 0', 'color': colors['text']}),
        html.P('Total number of purchase transactions', style={'margin': '0', 'color': '#888'})
    ]

    # Create Category Sales Bar Chart
    category_sales = filtered_df.groupby('Category')['Purchase_Amount'].sum().reset_index()
    category_sales = category_sales.sort_values('Purchase_Amount', ascending=False)

    category_fig = px.bar(
        category_sales,
        x='Category',
        y='Purchase_Amount',
        title='Sales by Category',
        color='Purchase_Amount',
        color_continuous_scale=px.colors.sequential.Blues,
        labels={'Purchase_Amount': 'Sales ($)', 'Category': 'Product Category'}
    )
    category_fig.update_layout(
        plot_bgcolor=colors['light_gray'],
        paper_bgcolor='white',
        font_color=colors['text'],
        xaxis={'categoryorder': 'total descending'}
    )

    # Create Payment Method Pie Chart
    payment_sales = filtered_df.groupby('Payment_Method')['Purchase_Amount'].sum().reset_index()
    payment_fig = px.pie(
        payment_sales,
        values='Purchase_Amount',
        names='Payment_Method',
        title='Sales by Payment Method',
        color_discrete_sequence=px.colors.sequential.Blues,
        hole=0.4
    )
    payment_fig.update_layout(
        plot_bgcolor=colors['light_gray'],
        paper_bgcolor='white',
        font_color=colors['text']
    )

    # Create Monthly Sales Trend Line Chart
    filtered_df['Month'] = filtered_df['Purchase_Date'].dt.strftime('%Y-%m')
    monthly_sales = filtered_df.groupby('Month')['Purchase_Amount'].sum().reset_index()
    monthly_sales = monthly_sales.sort_values('Month')

    monthly_fig = px.line(
        monthly_sales,
        x='Month',
        y='Purchase_Amount',
        title='Monthly Sales Trend',
        markers=True,
        labels={'Purchase_Amount': 'Sales ($)', 'Month': 'Month'}
    )
    monthly_fig.update_traces(line_color=colors['walmart_blue'])
    monthly_fig.update_layout(
        plot_bgcolor=colors['light_gray'],
        paper_bgcolor='white',
        font_color=colors['text']
    )

    # Create Gender Distribution Pie Chart
    gender_dist = filtered_df.groupby('Gender')['Customer_ID'].nunique().reset_index()
    gender_dist.columns = ['Gender', 'Count']

    gender_fig = px.pie(
        gender_dist,
        values='Count',
        names='Gender',
        title='Customer Gender Distribution',
        color_discrete_sequence=[colors['walmart_blue'], colors['walmart_yellow']],
        hole=0.4
    )
    gender_fig.update_layout(
        plot_bgcolor=colors['light_gray'],
        paper_bgcolor='white',
        font_color=colors['text']
    )

    # Create Age Distribution Histogram
    age_fig = px.histogram(
        filtered_df,
        x='Age',
        nbins=20,
        title='Customer Age Distribution',
        color_discrete_sequence=[colors['walmart_blue']],
        labels={'Age': 'Age', 'count': 'Number of Customers'}
    )
    age_fig.update_layout(
        plot_bgcolor=colors['light_gray'],
        paper_bgcolor='white',
        font_color=colors['text']
    )

    # Create Word Cloud for Products
    # Group by product and count occurrences
    product_counts = filtered_df['Product_Name'].value_counts().to_dict()

    # Generate the word cloud
    wc = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap='Blues',
        max_words=100
    ).generate_from_frequencies(product_counts)

    # Convert the word cloud to an image
    img = BytesIO()
    plt.figure(figsize=(10, 5))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.tight_layout(pad=0)
    plt.savefig(img, format='png')
    plt.close()
    img.seek(0)

    # Convert the image to base64 for displaying in the dashboard
    wordcloud_src = f"data:image/png;base64,{base64.b64encode(img.getvalue()).decode()}"

    return (
        total_sales_card,
        avg_purchase_card,
        unique_customers_card,
        total_orders_card,
        category_fig,
        payment_fig,
        monthly_fig,
        gender_fig,
        age_fig,
        wordcloud_src
    )

# Run the app
if __name__ == '__main__':
    app.run(debug=True)

<IPython.core.display.Javascript object>

In [6]:
import pandas as pd
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

# Load and prepare data
df = pd.read_csv("Walmart_customer_purchases.csv")
df['Purchase_Date'] = pd.to_datetime(df['Purchase_Date'])
df['Month'] = pd.to_datetime(df['Purchase_Date']).dt.month_name()
df['Year'] = pd.to_datetime(df['Purchase_Date']).dt.year

# Initialize app
app = Dash(__name__)
app.title = "Walmart Customer Dashboard"

# --- Dashboard Layouts ---
dashboard1_layout = html.Div(children=[
    html.H1('Walmart Customer Purchase Dashboard',
            style={'textAlign': 'center', 'color': '#503D36',
                   'font-size': 26}),
    html.Div([
        # Dropdown for Category selection
        html.Div([
            html.H2('Select Category:', style={'margin-right': '2em'}),
            dcc.Dropdown(df['Category'].unique(), value='Electronics', id='Category')
        ]),
        html.Div([
            html.Div([], id='plot1'),
            html.Div([], id='plot2')
        ], style={'display': 'flex'}),
    ])
])

dashboard2_layout = html.Div([
    html.H1('Dashboard 2', style={'textAlign': 'center', 'color': '#503D36', 'font-size': 26}),
    html.Div([
        html.Div([
            html.H2('Select Year:', style={'margin-right': '2em'}),
            dcc.Dropdown(df['Year'].unique(), value=2024, id='year')
        ]),
        html.Div([
            html.H2('Select city:', style={'margin-right': '2em'}),
            dcc.Dropdown(df['City'].unique(), value='Jeffreytown', id='city')
        ]),
        html.Div([], id='plot3'),
        html.Div([], id='plot4')
    ], style={'display': 'flex'}),
])

# --- Main App Layout ---
app.layout = html.Div([
    dcc.Tabs(id="tabs-example-graph", value='tab-1-example-graph', children=[
        dcc.Tab(label='Dashboard 1', value='tab-1-example-graph'),
        dcc.Tab(label='Dashboard 2', value='tab-2-example-graph'),
    ]),
    html.Div(id='tabs-content-example-graph')
])

# --- Callbacks ---
@app.callback(Output('tabs-content-example-graph', 'children'),
              Input('tabs-example-graph', 'value'))
def render_content(tab):
    if tab == 'tab-1-example-graph':
        return dashboard1_layout
    elif tab == 'tab-2-example-graph':
        return dashboard2_layout

@app.callback([Output(component_id='plot1', component_property='children'),
               Output(component_id='plot2', component_property='children')],
               [Input(component_id='Category', component_property='value')])
def update_dashboard1(input_Category):
    # data
    Category_data = df[df['Category'] == input_Category]
    summary_data = Category_data.groupby('Product_Name')['Purchase_Amount'].sum().sort_values(ascending=False).reset_index()

    # print(summary)
    fig1 = px.pie(summary_data, values='Purchase_Amount', names='Product_Name')
    fig1.update_traces(textposition='inside', textinfo='percent+label')
    fig2 = px.bar(summary_data, x='Product_Name', y='Purchase_Amount')

    return [dcc.Graph(figure=fig1),
            dcc.Graph(figure=fig2)]

@app.callback([Output(component_id='plot3', component_property='children'), # Updated callback for dashboard2
               Output(component_id='plot4', component_property='children')], # Updated callback for dashboard2
               [Input(component_id='city', component_property='value'),
                Input(component_id='year', component_property='value')])
def update_dashboard2(input_city, input_year): # Updated callback for dashboard2
    city_data = df[df['City'] == input_city]
    y_r_data = city_data[city_data['Year'] == input_year]
    est_data = y_r_data.groupby('Month')['Purchase_Amount'].mean().reset_index()
    fig3 = px.pie(est_data, values='Purchase_Amount', names='Month', title="{} : Monthly Average Purchase Amount in year {}".format(input_city, input_year))
    fig4 = px.bar(est_data, x='Month', y='Purchase_Amount', title="{} : Monthly Average Purchase Amount in year {}".format(input_city, input_year))
    return [dcc.Graph(figure=fig3), dcc.Graph(figure=fig4)]

# --- Run the app ---
if __name__ == '__main__':
    app.run()

<IPython.core.display.Javascript object>